# Semantic Link – Time Machine

**Git-style version control, diff, and debugging for Power BI semantic models.**

> "When a stakeholder says 'this KPI was right last Tuesday,' you currently have no way to prove what changed. Time Machine gives your semantic model a `git log` — snapshot its state on every refresh, diff any two points in history, and bisect to find the exact change that broke a measure's value."

---

## The problem

Semantic models evolve constantly — measures get tweaked, relationships change, columns are renamed, calculated columns get refactored. When a KPI silently starts returning the wrong number, developers have almost no tools to answer the most basic questions:

- *When* did this measure's value change?
- *What* change caused it?
- *What else* changed at the same time?
- *Can I see the model exactly as it was last Tuesday?*

Current workflows rely on memory, Git blame (if you're lucky enough to have Git integration), and manually re-running reports with old backup files.

## What Time Machine does

Time Machine treats semantic models like source code with a proper history:

1. **Capture** — Take a structured snapshot of the model (metadata + tracked measure values) with a single function call.
2. **Diff** — Compare any two snapshots to see exactly what changed in the model definition and in the measure outputs.
3. **Bisect** — Given a "good" snapshot and a "bad" snapshot, automatically binary-search to find the exact snapshot where a measure's value flipped — and explain which model change caused it.

All powered by `semantic-link` + `semantic-link-labs`, stored in Fabric-native Delta tables, runnable from any Fabric notebook.

## Why this wins developer experience

| Pain today | With Time Machine |
|---|---|
| "This number is wrong, but I don't know when it broke." | `bisect()` finds the exact snapshot. |
| "What did we change in the model between the March and April refreshes?" | `diff_snapshots('march_refresh', 'april_refresh')`. |
| "Did this measure's value really drift, or is it just me?" | Value history query on `tm_measure_values`. |
| "Can I see what the model looked like before this deployment?" | Query `tm_snapshots.tmsl_json` at any point. |


---

## Architecture at a glance

Time Machine has three layers: **Capture** reads the live model via Semantic Link, **Storage** persists structured snapshots in Delta, and **Analysis** offers diff / bisect / explain / plot on top.

```mermaid
flowchart TD
    M[Power BI Semantic Model] -->|sempy.fabric &#43; sempy_labs| CAP[capture_snapshot]
    CAP --> T1[(tm_snapshots<br/>TMSL &#43; counts)]
    CAP --> T2[(tm_measure_definitions<br/>DAX per measure)]
    CAP --> T3[(tm_measure_values<br/>evaluated values)]
    T1 --> DIFF[diff_snapshots]
    T2 --> DIFF
    T3 --> DIFF
    T1 --> BIS[bisect_measure_change]
    T3 --> BIS
    BIS --> EXP[explain_culprit]
    T2 --> EXP
    T3 --> PLOT[plot_measure_history]
    DIFF -.->|markdown report| USER((Developer))
    EXP  -.->|ranked suspects| USER
    PLOT -.->|timeline chart|  USER
```

**Capture** is the only operation that touches the live model — once you have snapshots on disk, everything downstream is pure Spark / pandas work against Delta.


---

## Step 1 — Configuration

Edit these values for your environment. If you're following the challenge's "getting started" steps with the Wide World Importers sample, you only need to change `WORKSPACE_NAME` and `DATASET_NAME`.

In [ ]:
# ---- USER CONFIG (edit this block) ---------------------------------------
WORKSPACE_NAME = "YourWorkspaceName"   # The Fabric workspace containing the model
DATASET_NAME   = "Retail Sample Model" # The semantic model name
LAKEHOUSE_NAME = "TimeMachineLakehouse" # Lakehouse to store Delta snapshots
# Measures you want Time Machine to track values for. Leave empty to track ALL.
TRACKED_MEASURES = [
    "Total Sales",
    "Total Profit",
    "Customer Count",
]
# Tolerance for flagging value drift (relative). 0.001 = 0.1%.
VALUE_DRIFT_TOLERANCE = 0.001
# ---- End user config -----------------------------------------------------

# Delta table paths (derived from config — usually no edit needed)
SNAPSHOT_TABLE      = f"Tables/tm_snapshots"
MEASURE_DEF_TABLE   = f"Tables/tm_measure_definitions"
MEASURE_VALUE_TABLE = f"Tables/tm_measure_values"


## Step 2 — Install and import

`semantic-link` and `semantic-link-labs` are pre-installed in most Fabric runtimes. `deepdiff` powers the structural diff engine.

In [ ]:
%pip install semantic-link-labs deepdiff --quiet

In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
import json
import uuid
import hashlib
import re
from datetime import datetime, timezone
from deepdiff import DeepDiff
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    IntegerType, DoubleType, LongType
)

# Get workspace + dataset IDs up front so we don't re-resolve them every call
WORKSPACE_ID = fabric.resolve_workspace_id(WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"Dataset:   {DATASET_NAME}")


## Step 3 — Create the Delta tables

Three tables store the full history:

- **`tm_snapshots`** — one row per snapshot, contains the full TMSL (BIM) JSON and summary counts.
- **`tm_measure_definitions`** — DAX expression of every measure at that point in time (makes diffs queryable without parsing JSON).
- **`tm_measure_values`** — computed values of tracked measures (enables drift detection and bisect).

Run this cell once per lakehouse. Subsequent runs are no-ops.

In [ ]:
snapshots_schema = StructType([
    StructField("snapshot_id",        StringType(),    False),
    StructField("timestamp",          TimestampType(), False),
    StructField("workspace_id",       StringType(),    False),
    StructField("workspace_name",     StringType(),    False),
    StructField("dataset_name",       StringType(),    False),
    StructField("label",              StringType(),    True),
    StructField("tmsl_json",          StringType(),    True),
    StructField("tmsl_hash",          StringType(),    True),
    StructField("measure_count",      IntegerType(),   True),
    StructField("table_count",        IntegerType(),   True),
    StructField("column_count",       IntegerType(),   True),
    StructField("relationship_count", IntegerType(),   True),
    StructField("captured_by",        StringType(),    True),
])

measure_def_schema = StructType([
    StructField("snapshot_id",    StringType(), False),
    StructField("measure_name",   StringType(), False),
    StructField("table_name",     StringType(), True),
    StructField("expression",     StringType(), True),
    StructField("format_string",  StringType(), True),
    StructField("display_folder", StringType(), True),
    StructField("description",    StringType(), True),
])

measure_value_schema = StructType([
    StructField("snapshot_id",     StringType(),    False),
    StructField("measure_name",    StringType(),    False),
    StructField("grouping_json",   StringType(),    False),
    StructField("value_numeric",   DoubleType(),    True),
    StructField("value_string",    StringType(),    True),
    StructField("evaluated_at",    TimestampType(), False),
])

def _ensure_table(path, schema):
    try:
        spark.read.format("delta").load(path).limit(0).count()
    except Exception:
        (spark.createDataFrame([], schema)
             .write.format("delta").mode("overwrite").save(path))
        print(f"Created {path}")

_ensure_table(SNAPSHOT_TABLE,      snapshots_schema)
_ensure_table(MEASURE_DEF_TABLE,   measure_def_schema)
_ensure_table(MEASURE_VALUE_TABLE, measure_value_schema)
print("Delta tables ready.")


---

## Step 4 — Capture layer

`capture_snapshot()` is the single entry point. It pulls the full model definition (TMSL) via `sempy-labs`, enumerates structural elements via `sempy.fabric`, evaluates tracked measures, and writes all three Delta tables atomically.

### Design notes

- **TMSL via `labs.get_semantic_model_bim()`** — returns the authoritative BIM JSON. We hash it so duplicate snapshots (no changes since last capture) are cheap to detect.
- **Measure values at total level** — evaluated via `fabric.evaluate_measure()` with no groupby. Extend the dimensional breakdowns in `_evaluate_tracked_measures` for richer drift detection.
- **Every snapshot gets a UUID** — human labels (`label` field) are optional but recommended for demo-worthy snapshots like `"pre-refresh-2026-04-14"`.

In [ ]:
def _current_user():
    try:
        return spark.sparkContext.sparkUser()
    except Exception:
        return "unknown"

def _extract_measures_from_tmsl(tmsl_obj):
    """Parse measures directly out of the TMSL (BIM) JSON.

    This is the authoritative source and always reflects the just-retrieved
    model definition. Using ``fabric.list_measures()`` instead can return
    stale DAX expressions from a service-side cache, which breaks diff
    correctness after a measure edit. Parse TMSL and we always see the
    real current state.

    Returns:
        pandas.DataFrame with columns matching what the rest of the pipeline
        expects: Measure Name, Table Name, Measure Expression, Format String,
        Display Folder, Description.
    """
    rows = []
    tables = tmsl_obj.get("model", {}).get("tables", [])
    for t in tables:
        t_name = t.get("name")
        for m in t.get("measures", []):
            expr = m.get("expression")
            # TMSL sometimes stores multi-line DAX as a list of strings
            if isinstance(expr, list):
                expr = "\n".join(expr)
            rows.append({
                "Measure Name":       m.get("name"),
                "Table Name":         t_name,
                "Measure Expression": expr,
                "Format String":      m.get("formatString"),
                "Display Folder":     m.get("displayFolder"),
                "Description":        m.get("description"),
            })
    return pd.DataFrame(rows)

def _capture_metadata():
    """Pull the full TMSL + structural element counts from the live model.

    Uses ``sempy_labs.get_semantic_model_bim`` for the authoritative TMSL,
    then derives the measure list from the TMSL directly (see the note on
    ``_extract_measures_from_tmsl``). Tables / columns / relationships are
    counted via ``sempy.fabric`` — those endpoints are not affected by the
    measure caching issue.
    """
    tmsl = labs.get_semantic_model_bim(
        dataset=DATASET_NAME,
        workspace=WORKSPACE_NAME,
    )
    # get_semantic_model_bim may return dict OR json string depending on version
    tmsl_str = tmsl if isinstance(tmsl, str) else json.dumps(tmsl, indent=2)
    tmsl_obj = tmsl if isinstance(tmsl, dict) else json.loads(tmsl)
    tmsl_hash = hashlib.sha256(tmsl_str.encode("utf-8")).hexdigest()[:16]

    # Authoritative, uncached measure list
    measures_df      = _extract_measures_from_tmsl(tmsl_obj)
    tables_df        = fabric.list_tables(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
    columns_df       = fabric.list_columns(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
    relationships_df = fabric.list_relationships(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)

    return {
        "tmsl_json":          tmsl_str,
        "tmsl_obj":           tmsl_obj,
        "tmsl_hash":          tmsl_hash,
        "measures":           measures_df,
        "measure_count":      len(measures_df),
        "table_count":        len(tables_df),
        "column_count":       len(columns_df),
        "relationship_count": len(relationships_df),
    }

def _evaluate_tracked_measures(measures_df):
    """Evaluate the configured TRACKED_MEASURES (or all measures, if empty).

    Numeric coercion is defensive: ``evaluate_measure`` can return numpy
    scalars, pandas Int64, Decimal, or Python floats depending on the measure
    and runtime. We attempt ``float()`` in a try/except and fall back to the
    string representation — this fixes the Int64 case (e.g. ``Total Quantity``)
    that an ``isinstance(val, (int, float))`` check silently rejects.
    """
    targets = TRACKED_MEASURES if TRACKED_MEASURES else measures_df["Measure Name"].tolist()
    rows = []
    now = datetime.now(timezone.utc)
    for name in targets:
        try:
            result = fabric.evaluate_measure(
                dataset=DATASET_NAME,
                workspace=WORKSPACE_NAME,
                measure=name,
            )
            val = result.iloc[0, 0] if len(result) else None
            numeric, string = None, None
            if val is not None and not pd.isna(val):
                try:
                    numeric = float(val)
                except (TypeError, ValueError):
                    string = str(val)
            rows.append({
                "measure_name":  name,
                "grouping_json": "{}",
                "value_numeric": numeric,
                "value_string":  string,
                "evaluated_at":  now,
            })
        except Exception as e:
            print(f"  [warn] could not evaluate '{name}': {e}")
            rows.append({
                "measure_name":  name,
                "grouping_json": "{}",
                "value_numeric": None,
                "value_string":  f"ERROR: {e}",
                "evaluated_at":  now,
            })
    return rows

def capture_snapshot(label: str = None) -> str:
    """Capture a full snapshot of the configured semantic model.

    Pulls the live TMSL, enumerates measures / tables / columns / relationships,
    evaluates the configured tracked measures, and appends rows to the three
    Delta tables. This is the only operation that touches the live model — all
    diff / bisect / plot operations run against the persisted snapshots.

    Args:
        label: Optional human-readable tag for the snapshot (e.g.
               ``"pre-refresh-march-15"``). Stored alongside the snapshot
               and surfaced in diff and plot output.

    Returns:
        The snapshot_id (UUID string). Hold onto this to diff or bisect later.

    Example:
        >>> baseline_id = capture_snapshot(label="baseline")
        >>> # ... edit the model in Power BI ...
        >>> after_id    = capture_snapshot(label="after-march-edit")
        >>> diff_snapshots(baseline_id, after_id)
    """
    snap_id = str(uuid.uuid4())
    now     = datetime.now(timezone.utc)
    print(f"[{now.isoformat()}] Capturing snapshot {snap_id[:8]}...")

    meta   = _capture_metadata()
    values = _evaluate_tracked_measures(meta["measures"])

    # Row 1: tm_snapshots
    snap_row = Row(
        snapshot_id        = snap_id,
        timestamp          = now,
        workspace_id       = WORKSPACE_ID,
        workspace_name     = WORKSPACE_NAME,
        dataset_name       = DATASET_NAME,
        label              = label,
        tmsl_json          = meta["tmsl_json"],
        tmsl_hash          = meta["tmsl_hash"],
        measure_count      = meta["measure_count"],
        table_count        = meta["table_count"],
        column_count       = meta["column_count"],
        relationship_count = meta["relationship_count"],
        captured_by        = _current_user(),
    )
    (spark.createDataFrame([snap_row], snapshots_schema)
         .write.format("delta").mode("append").save(SNAPSHOT_TABLE))

    # Rows: tm_measure_definitions
    md_rows = []
    for _, m in meta["measures"].iterrows():
        md_rows.append(Row(
            snapshot_id    = snap_id,
            measure_name   = m.get("Measure Name"),
            table_name     = m.get("Table Name"),
            expression     = m.get("Measure Expression") or m.get("Expression"),
            format_string  = m.get("Format String"),
            display_folder = m.get("Display Folder"),
            description    = m.get("Description"),
        ))
    if md_rows:
        (spark.createDataFrame(md_rows, measure_def_schema)
             .write.format("delta").mode("append").save(MEASURE_DEF_TABLE))

    # Rows: tm_measure_values
    mv_rows = [Row(snapshot_id=snap_id, **v) for v in values]
    if mv_rows:
        (spark.createDataFrame(mv_rows, measure_value_schema)
             .write.format("delta").mode("append").save(MEASURE_VALUE_TABLE))

    print(f"  ✓ saved: {meta['measure_count']} measures, "
          f"{meta['table_count']} tables, "
          f"{meta['column_count']} columns, "
          f"{meta['relationship_count']} relationships")
    print(f"  ✓ tracked values: {len(values)}")
    print(f"  snapshot_id = {snap_id}")
    return snap_id


---

## Step 5 — Diff engine

Given two snapshot IDs, the diff engine produces two reports:

1. **Metadata diff** — every structural change (added/removed/modified measures, columns, relationships) with a classification (`breaking` / `non-breaking` / `additive`).
2. **Value diff** — every tracked measure whose value drifted beyond tolerance.

The hard part is making the output *readable*. Raw `deepdiff` output looks like debugger vomit; we post-process it into a clean, categorized report.

In [ ]:
def _load_snapshot(snap_id):
    """Load a single snapshot's row from tm_snapshots."""
    df = spark.read.format("delta").load(SNAPSHOT_TABLE).filter(f"snapshot_id = '{snap_id}'")
    rows = df.collect()
    if not rows:
        raise ValueError(f"No snapshot found with id {snap_id}")
    return rows[0].asDict()

def _load_measure_defs(snap_id):
    return (spark.read.format("delta").load(MEASURE_DEF_TABLE)
                 .filter(f"snapshot_id = '{snap_id}'")
                 .toPandas())

def _load_measure_values(snap_id):
    return (spark.read.format("delta").load(MEASURE_VALUE_TABLE)
                 .filter(f"snapshot_id = '{snap_id}'")
                 .toPandas())

# Classification rules. Extend as needed.
_BREAKING = {
    "deleted_measure", "deleted_column", "deleted_table",
    "deleted_relationship", "changed_column_datatype",
    "modified_dax",  # DAX change is always potentially value-breaking
}
_ADDITIVE = {
    "added_measure", "added_column", "added_table",
    "added_relationship", "added_calc_group",
}

def _classify(change_kind):
    if change_kind in _BREAKING: return "breaking"
    if change_kind in _ADDITIVE: return "additive"
    return "non-breaking"

def diff_metadata(snap_a_id, snap_b_id):
    """Structured diff of two snapshots' model definitions."""
    defs_a = _load_measure_defs(snap_a_id).set_index("measure_name")
    defs_b = _load_measure_defs(snap_b_id).set_index("measure_name")

    changes = []

    added   = set(defs_b.index) - set(defs_a.index)
    deleted = set(defs_a.index) - set(defs_b.index)
    common  = set(defs_a.index) & set(defs_b.index)

    for m in sorted(added):
        changes.append({"kind": "added_measure",   "object": m,
                        "classification": _classify("added_measure"),
                        "before": None, "after": defs_b.loc[m, "expression"]})
    for m in sorted(deleted):
        changes.append({"kind": "deleted_measure", "object": m,
                        "classification": _classify("deleted_measure"),
                        "before": defs_a.loc[m, "expression"], "after": None})
    for m in sorted(common):
        if defs_a.loc[m, "expression"] != defs_b.loc[m, "expression"]:
            changes.append({"kind": "modified_dax", "object": m,
                            "classification": _classify("modified_dax"),
                            "before": defs_a.loc[m, "expression"],
                            "after":  defs_b.loc[m, "expression"]})

    # Full TMSL diff for structural changes beyond measures
    snap_a = _load_snapshot(snap_a_id)
    snap_b = _load_snapshot(snap_b_id)
    tmsl_a = json.loads(snap_a["tmsl_json"])
    tmsl_b = json.loads(snap_b["tmsl_json"])
    dd = DeepDiff(tmsl_a, tmsl_b, ignore_order=True, verbose_level=0)
    # Summary counts for the TMSL-level changes (measures already itemized above)
    tmsl_summary = {
        "items_added":     len(dd.get("iterable_item_added", {})) + len(dd.get("dictionary_item_added", set())),
        "items_removed":   len(dd.get("iterable_item_removed", {})) + len(dd.get("dictionary_item_removed", set())),
        "values_changed":  len(dd.get("values_changed", {})),
        "type_changes":    len(dd.get("type_changes", {})),
    }

    return {
        "from_snapshot":   snap_a_id,
        "to_snapshot":     snap_b_id,
        "measure_changes": changes,
        "tmsl_summary":    tmsl_summary,
        "raw_deepdiff":    dd.to_dict() if dd else {},
    }

def diff_values(snap_a_id, snap_b_id, tolerance=None):
    """Value-level diff of tracked measures between two snapshots."""
    if tolerance is None:
        tolerance = VALUE_DRIFT_TOLERANCE
    vals_a = _load_measure_values(snap_a_id)
    vals_b = _load_measure_values(snap_b_id)
    merged = vals_a.merge(
        vals_b, on=["measure_name", "grouping_json"],
        suffixes=("_a", "_b"), how="outer", indicator=True,
    )
    drifts = []
    for _, r in merged.iterrows():
        a, b = r.get("value_numeric_a"), r.get("value_numeric_b")
        if pd.isna(a) and pd.isna(b): continue
        if pd.isna(a) or pd.isna(b):
            drifts.append({"measure": r["measure_name"], "grouping": r["grouping_json"],
                           "before": None if pd.isna(a) else a,
                           "after":  None if pd.isna(b) else b,
                           "rel_change": None, "classification": "appeared_or_disappeared"})
            continue
        if a == 0 and b == 0: continue
        rel = abs(b - a) / max(abs(a), 1e-12)
        if rel > tolerance:
            drifts.append({"measure": r["measure_name"], "grouping": r["grouping_json"],
                           "before": a, "after": b, "rel_change": rel,
                           "classification": "drifted"})
    return drifts

def format_diff_report(meta_diff, value_diff):
    """Return a human-readable markdown report."""
    lines = []
    lines.append(f"### Time Machine Diff Report")
    lines.append(f"**From:** `{meta_diff['from_snapshot'][:8]}`  →  **To:** `{meta_diff['to_snapshot'][:8]}`\n")

    # Measure-level changes
    mc = meta_diff["measure_changes"]
    breaking  = [c for c in mc if c["classification"] == "breaking"]
    additive  = [c for c in mc if c["classification"] == "additive"]
    other     = [c for c in mc if c["classification"] == "non-breaking"]

    lines.append(f"**Measure changes:** {len(mc)} "
                 f"({len(breaking)} breaking, {len(additive)} additive, {len(other)} other)")

    if breaking:
        lines.append("\n#### ⚠ Breaking changes")
        for c in breaking:
            lines.append(f"- `{c['kind']}` on **{c['object']}**")
            if c["kind"] == "modified_dax":
                lines.append(f"  - **Before:** `{c['before']}`")
                lines.append(f"  - **After:**  `{c['after']}`")
    if additive:
        lines.append("\n#### ＋ Additive changes")
        for c in additive:
            lines.append(f"- `{c['kind']}` on **{c['object']}**")
    if other:
        lines.append("\n#### Other")
        for c in other:
            lines.append(f"- `{c['kind']}` on **{c['object']}**")

    # Value drift
    if value_diff:
        lines.append(f"\n### Value drift (tolerance {VALUE_DRIFT_TOLERANCE:.3%})")
        for d in value_diff:
            rc = f"{d['rel_change']:.3%}" if d.get("rel_change") is not None else "n/a"
            lines.append(f"- **{d['measure']}** ({d['classification']}): "
                         f"{d['before']} → {d['after']} "
                         f"(Δ {rc})")
    else:
        lines.append("\n*No tracked measure values drifted beyond tolerance.*")

    # TMSL-level summary
    ts = meta_diff["tmsl_summary"]
    lines.append(f"\n### TMSL-level summary")
    lines.append(f"Items added: {ts['items_added']} | removed: {ts['items_removed']} | "
                 f"values changed: {ts['values_changed']} | type changes: {ts['type_changes']}")

    return "\n".join(lines)

def diff_snapshots(snap_a_id, snap_b_id, render=True):
    """Compare two snapshots and return a structured + rendered diff.

    Produces both a **metadata diff** (added / deleted / modified measures,
    TMSL structural summary) and a **value diff** (measure values that
    drifted beyond ``VALUE_DRIFT_TOLERANCE``). The returned dict gives you
    the raw structures for programmatic use; set ``render=True`` for the
    inline markdown report.

    Args:
        snap_a_id: Earlier snapshot_id ("from").
        snap_b_id: Later snapshot_id ("to").
        render:    If True, displays a markdown report inline.

    Returns:
        dict with keys:
            - ``"metadata"``: output of ``diff_metadata``
            - ``"values"``:   list of drift dicts from ``diff_values``
            - ``"report"``:   the markdown report string

    Example:
        >>> diff = diff_snapshots(baseline_id, broken_id)
        >>> [c for c in diff["metadata"]["measure_changes"] if c["classification"] == "breaking"]
    """
    m = diff_metadata(snap_a_id, snap_b_id)
    v = diff_values(snap_a_id, snap_b_id)
    report = format_diff_report(m, v)
    if render:
        from IPython.display import Markdown, display
        display(Markdown(report))
    return {"metadata": m, "values": v, "report": report}


---

## Step 6 — Bisect: the killer feature

Given a measure that's "good at snapshot A" and "bad at snapshot Z," with an arbitrary number of snapshots in between, automatically find the one snapshot that flipped the value — and explain which model change was responsible.

### Algorithm

1. Pull all snapshot IDs between good and bad, ordered by timestamp.
2. Binary search: at the midpoint, read the measure's value from `tm_measure_values`.
3. If midpoint matches "good", the flip is in the later half. If it matches "bad", it's in the earlier half.
4. Recurse until we've isolated a single snapshot.
5. Run `diff_metadata()` between that snapshot and its immediate predecessor — the culprit change is in there.
6. Score each diff by relevance to the target measure (direct edit > dependency edit > unrelated edit) and return ranked.

### Why binary search matters

With 50 daily snapshots, linear scan = 50 value evaluations. Bisect = ~6. Since value reads are cached in Delta, not re-evaluated, this is pure UX — but it also keeps the demo tight.

In [ ]:
def _ordered_snapshots_between(snap_a_id, snap_b_id):
    """All snapshot rows with timestamp in [ts(a), ts(b)], sorted."""
    snap_a = _load_snapshot(snap_a_id)
    snap_b = _load_snapshot(snap_b_id)
    t_lo, t_hi = sorted([snap_a["timestamp"], snap_b["timestamp"]])
    df = (spark.read.format("delta").load(SNAPSHOT_TABLE)
            .filter(f"timestamp >= '{t_lo.isoformat()}' AND timestamp <= '{t_hi.isoformat()}'")
            .filter(f"dataset_name = '{DATASET_NAME}'")
            .orderBy("timestamp")
            .toPandas())
    return df

def _measure_value_at(snap_id, measure_name):
    df = (spark.read.format("delta").load(MEASURE_VALUE_TABLE)
            .filter(f"snapshot_id = '{snap_id}' AND measure_name = '{measure_name}' "
                    f"AND grouping_json = '{{}}'")
            .toPandas())
    if df.empty:
        return None
    return df.iloc[0]["value_numeric"]

def _values_match(a, b, tolerance=None):
    if tolerance is None: tolerance = VALUE_DRIFT_TOLERANCE
    if a is None and b is None: return True
    if a is None or b is None:  return False
    if a == 0 and b == 0: return True
    return abs(b - a) / max(abs(a), 1e-12) <= tolerance

def bisect_measure_change(measure_name, good_snap_id, bad_snap_id, verbose=True):
    """Binary-search for the snapshot that flipped a measure's value.

    Given a snapshot where the measure value was correct and a later snapshot
    where it is wrong, this finds the exact snapshot where it flipped. Uses
    the values already evaluated at capture time (no live queries against the
    model), so it is effectively free to run.

    Args:
        measure_name:  Name of the measure to bisect on (must be tracked).
        good_snap_id:  snapshot_id where the value was correct.
        bad_snap_id:   snapshot_id where the value is wrong.
        verbose:       Print each probe step.

    Returns:
        dict with the culprit + its predecessor, or ``None`` if the values
        already match (nothing to find):
            - ``"culprit_snapshot"``: first snapshot where value flipped
            - ``"predecessor_snapshot"``: the snapshot immediately before
            - ``"good_value"``, ``"bad_value"``: the two boundary values

    Example:
        >>> result = bisect_measure_change("Total Sales", baseline_id, broken_id)
        >>> explain_culprit(result["culprit_snapshot"],
        ...                 result["predecessor_snapshot"],
        ...                 measure_name="Total Sales")
    """
    snaps = _ordered_snapshots_between(good_snap_id, bad_snap_id)
    if len(snaps) < 2:
        raise ValueError("Need at least 2 snapshots between good and bad.")

    good_val = _measure_value_at(good_snap_id, measure_name)
    bad_val  = _measure_value_at(bad_snap_id,  measure_name)

    if _values_match(good_val, bad_val):
        if verbose: print(f"Values already match ({good_val}). No flip to find.")
        return None

    if verbose:
        print(f"Bisecting '{measure_name}': {good_val} (good) → {bad_val} (bad) "
              f"across {len(snaps)} snapshots")

    lo, hi = 0, len(snaps) - 1
    # Find first index where value matches "bad"
    while lo < hi:
        mid = (lo + hi) // 2
        mid_snap = snaps.iloc[mid]["snapshot_id"]
        mid_val  = _measure_value_at(mid_snap, measure_name)
        matches_good = _values_match(mid_val, good_val)
        if verbose:
            print(f"  probe @ {mid_snap[:8]}: value={mid_val} → "
                  f"{'good side' if matches_good else 'bad side'}")
        if matches_good:
            lo = mid + 1
        else:
            hi = mid

    culprit_idx = lo
    culprit_snap = snaps.iloc[culprit_idx]["snapshot_id"]
    predecessor  = snaps.iloc[culprit_idx - 1]["snapshot_id"] if culprit_idx > 0 else good_snap_id

    if verbose:
        print(f"\n>>> Culprit snapshot: {culprit_snap}")
        print(f"    (predecessor:     {predecessor})")
    return {
        "culprit_snapshot":     culprit_snap,
        "predecessor_snapshot": predecessor,
        "good_value":           good_val,
        "bad_value":            bad_val,
    }

def _extract_referenced_objects(dax_expression):
    """Very light DAX parser: finds [Column] and 'Table' references."""
    if not dax_expression: return set()
    refs = set()
    # 'Table'[Column] or [Column]
    for m in re.findall(r"'?([A-Za-z0-9 _]+?)'?\[([A-Za-z0-9 _]+)\]", dax_expression):
        refs.add(m[0].strip()); refs.add(m[1].strip())
    for m in re.findall(r"\[([A-Za-z0-9 _]+)\]", dax_expression):
        refs.add(m.strip())
    return refs

def explain_culprit(culprit_snap_id, predecessor_snap_id, measure_name):
    """Rank a snapshot's metadata changes by relevance to a target measure.

    Runs a metadata diff between the culprit and its predecessor, then scores
    each change:
        * **100 (DIRECT)**     — the target measure itself was modified
        * **50  (DEPENDENCY)** — a measure/column referenced by the target
          was modified (extracted via a light regex-based DAX parser)
        * **1   (peripheral)** — everything else

    Args:
        culprit_snap_id:      snapshot_id returned by ``bisect_measure_change``.
        predecessor_snap_id:  The snapshot immediately before the culprit.
        measure_name:         The measure whose value flipped.

    Returns:
        A list of change dicts with an added ``relevance_score`` field,
        sorted by score descending. Also renders a ranked suspect list inline.
    """
    meta_diff = diff_metadata(predecessor_snap_id, culprit_snap_id)

    # Get the culprit snapshot's definition of the target measure to see dependencies
    defs = _load_measure_defs(culprit_snap_id).set_index("measure_name")
    target_expr = defs.loc[measure_name, "expression"] if measure_name in defs.index else ""
    deps = _extract_referenced_objects(target_expr)

    ranked = []
    for c in meta_diff["measure_changes"]:
        score = 0
        if c["object"] == measure_name:        score += 100  # direct hit
        elif c["object"] in deps:              score += 50   # dependency edit
        else:                                  score += 1
        ranked.append({**c, "relevance_score": score})
    ranked.sort(key=lambda x: -x["relevance_score"])

    from IPython.display import Markdown, display
    lines = [f"### Root-cause analysis: `{measure_name}`\n"]
    lines.append(f"Culprit snapshot: `{culprit_snap_id[:8]}`  ←  Predecessor: `{predecessor_snap_id[:8]}`\n")
    lines.append("**Suspects (ranked by relevance to this measure):**\n")
    for r in ranked[:10]:
        tag = "🎯 DIRECT" if r["relevance_score"] >= 100 else \
              "🔗 DEPENDENCY" if r["relevance_score"] >= 50 else "· peripheral"
        lines.append(f"- {tag} `{r['kind']}` on **{r['object']}**")
        if r["kind"] == "modified_dax":
            lines.append(f"  - before: `{r['before']}`")
            lines.append(f"  - after:  `{r['after']}`")
    display(Markdown("\n".join(lines)))
    return ranked


---

## Step 7 — Demo walkthrough

This is the story the judges will see. Run each cell in order.

### Scene 1: Take a baseline snapshot

You've just finished building your model and trust all its numbers. Capture a baseline.

In [ ]:
baseline_id = capture_snapshot(label="baseline")


### Scene 2: A refresh happens, you tweak a measure

Imagine a day passes. Someone edits `[Total Sales]` in the model — maybe they "fixed" it to exclude returns. Capture a new snapshot.

In [ ]:
# ACTION REQUIRED FOR DEMO:
# 1. Open the semantic model in the Fabric portal.
# 2. Edit the `[Total Sales]` measure (e.g., wrap it in CALCULATE(..., FILTER(...)) to subtly change its result).
# 3. Save the model.
# 4. Run this cell.
snap_v2 = capture_snapshot(label="after-total-sales-edit")


### Scene 3: More time passes, more changes

More days go by, more tweaks happen. Capture a few more snapshots, each after a small model change.

In [ ]:
# Repeat the pattern: edit the model, capture a snapshot. Label them meaningfully.
snap_v3 = capture_snapshot(label="after-new-profit-measure")
# ... (for full demo, capture 5+ snapshots with different changes)
snap_v4 = capture_snapshot(label="current")


### Scene 4: Someone says the number looks wrong

A stakeholder flags that `[Total Sales]` seems too low compared to last month. Diff baseline vs. current:

In [ ]:
diff = diff_snapshots(baseline_id, snap_v4)


### Scene 5: Bisect to the exact change

The diff shows multiple changes between baseline and current. Which one actually changed the number? Bisect.

In [ ]:
result = bisect_measure_change(
    measure_name="Total Sales",
    good_snap_id=baseline_id,
    bad_snap_id=snap_v4,
)


### Scene 6: Root-cause explanation

Now rank the changes in the culprit snapshot by relevance to `[Total Sales]`:

In [ ]:
if result:
    ranked = explain_culprit(
        culprit_snap_id      = result["culprit_snapshot"],
        predecessor_snap_id  = result["predecessor_snapshot"],
        measure_name         = "Total Sales",
    )


---

## Step 7 — Visualize a measure's history

Bisect tells you exactly where something broke. Sometimes you also want a bird's-eye view: **how has this measure behaved across every snapshot I've ever taken?** `plot_measure_history()` renders a timeline annotated with automatic DRIFT markers.


In [ ]:
# -------------------------------------------------------------------
# Visualization: measure value over time across all snapshots
# -------------------------------------------------------------------
def plot_measure_history(measure_name: str,
                         dataset_name: str = None,
                         drift_tolerance: float = None,
                         figsize=(11, 4)):
    """Plot a measure's evaluated value across every snapshot in chronological order.

    Visualizes how a tracked measure has drifted over the history of captured
    snapshots. Any snapshot where the value changed more than `drift_tolerance`
    versus the *previous* snapshot is annotated with a red DRIFT marker.

    Args:
        measure_name:     Name of the measure to plot (must be in TRACKED_MEASURES).
        dataset_name:     Optional dataset override. Defaults to DATASET_NAME.
        drift_tolerance:  Relative change threshold for marking drift. Defaults
                          to VALUE_DRIFT_TOLERANCE from config.
        figsize:          matplotlib figure size tuple.

    Returns:
        pandas.DataFrame with columns [snapshot_id, label, timestamp, value]
        ordered chronologically. Also renders the chart inline.

    Example:
        >>> plot_measure_history("Total Sales")
    """
    import matplotlib.pyplot as plt
    if dataset_name is None: dataset_name = DATASET_NAME
    if drift_tolerance is None: drift_tolerance = VALUE_DRIFT_TOLERANCE

    snaps = (spark.read.format("delta").load(SNAPSHOT_TABLE)
                .filter(f"dataset_name = '{dataset_name}'")
                .select("snapshot_id", "label", "timestamp")
                .orderBy("timestamp")
                .toPandas())
    if snaps.empty:
        print(f"No snapshots found for dataset '{dataset_name}'.")
        return snaps

    vals = (spark.read.format("delta").load(MEASURE_VALUE_TABLE)
                .filter(f"measure_name = '{measure_name}' AND grouping_json = '{{}}'")
                .select("snapshot_id", "value_numeric")
                .toPandas())

    df = snaps.merge(vals, on="snapshot_id", how="left").rename(
        columns={"value_numeric": "value"})

    # Mark drift versus the previous snapshot
    df["prev"]    = df["value"].shift(1)
    def _drifted(r):
        a, b = r["prev"], r["value"]
        if pd.isna(a) or pd.isna(b): return False
        if a == 0 and b == 0: return False
        return abs(b - a) / max(abs(a), 1e-12) > drift_tolerance
    df["drift"] = df.apply(_drifted, axis=1)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df["timestamp"], df["value"], marker="o", linewidth=2)
    for _, r in df[df["drift"]].iterrows():
        ax.annotate("DRIFT", xy=(r["timestamp"], r["value"]),
                    xytext=(0, 15), textcoords="offset points",
                    color="crimson", fontsize=9, fontweight="bold",
                    ha="center",
                    arrowprops=dict(arrowstyle="->", color="crimson"))
    # Annotate labels under each point
    for _, r in df.iterrows():
        if r["label"]:
            ax.annotate(r["label"], xy=(r["timestamp"], r["value"]),
                        xytext=(0, -22), textcoords="offset points",
                        fontsize=7, ha="center", color="#555", rotation=0)
    ax.set_title(f"'{measure_name}' value history — {len(df)} snapshots")
    ax.set_xlabel("Snapshot timestamp")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    return df[["snapshot_id", "label", "timestamp", "value", "drift"]]


In [ ]:
# Timeline for any tracked measure
plot_measure_history("Total Sales")


---

## How to adopt Time Machine in YOUR workspace

1. **Clone this notebook** into any Fabric workspace.
2. **Edit the config cell** at the top — set `WORKSPACE_NAME`, `DATASET_NAME`, and the measures you want to track.
3. **Run Step 3** once to create the Delta tables in your lakehouse.
4. **Call `capture_snapshot()`** — manually for now, or wrap it in a Data Pipeline to run on a schedule.
5. **When something looks wrong**, call `diff_snapshots()` or `bisect_measure_change()` from any notebook.

### Suggested automation patterns

- **Pre-refresh snapshot:** trigger `capture_snapshot("pre-refresh")` as the first step of every dataset refresh pipeline.
- **Pre-deploy snapshot:** trigger before any Fabric Git integration push or deployment pipeline stage.
- **Scheduled baseline:** daily cron that captures `capture_snapshot(f"daily-{date.today()}")`.
- **Alerting:** after each capture, run `diff_values()` against the previous snapshot; post to Teams/Slack if drift > threshold.

---

## Known limitations

- **Value snapshots are point-in-time:** if the underlying fact data changes, a measure can "drift" even when the model definition didn't change. Time Machine detects this; it doesn't distinguish model drift from data drift. *Future work: snapshot fact-table row counts too.*
- **Dimensional breakdowns are total-only in this MVP:** extend `_evaluate_tracked_measures` with user-defined groupby columns for richer drift surfaces.
- **DAX dependency extraction is regex-based:** misses nested column references inside CALCULATETABLE, etc. A real AST-based DAX parser (TOM) would be more accurate.
- **Single-dataset per notebook config:** to Time-Machine multiple datasets, clone the config cell and namespace the Delta table paths.

## Extension ideas

- **Storage-side:** add a `tm_column_values_histogram` table snapshotting fact-column cardinality/null-rate per snapshot — surfaces data drift.
- **UI:** wrap the diff + bisect functions in a small Streamlit/Dash app for non-notebook users.
- **Git integration:** auto-commit TMSL JSON to a Git branch on each snapshot, so you get a real `git log`.
- **Scheduled monitor:** Fabric Data Pipeline that captures + diffs + alerts automatically.

---

## Credits & citations

- **semantic-link** — https://github.com/microsoft/semantic-link (`sempy`)
- **semantic-link-labs** — https://github.com/microsoft/semantic-link-labs (`sempy_labs`)
- **deepdiff** — https://github.com/seperman/deepdiff (structural object diffing)
- **Wide World Importers sample** — Microsoft Fabric built-in sample dataset

---

*Submitted to the Fabric Semantic Link Developer Experience Challenge, April 2026.*
